In [1]:
import polars as pl
from pathlib import Path
from csrio_image2biomass.configs.settings import RAW_DATA_DIR, AUGUMENTED_DATA_DIR

train = pl.read_csv(RAW_DATA_DIR / "train.csv")
test = pl.read_csv(RAW_DATA_DIR / "test.csv")
train

sample_id,image_path,Sampling_Date,State,Species,Pre_GSHH_NDVI,Height_Ave_cm,target_name,target
str,str,str,str,str,f64,f64,str,f64
"""ID1011485656__Dry_Clover_g""","""train/ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,"""Dry_Clover_g""",0.0
"""ID1011485656__Dry_Dead_g""","""train/ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,"""Dry_Dead_g""",31.9984
"""ID1011485656__Dry_Green_g""","""train/ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,"""Dry_Green_g""",16.2751
"""ID1011485656__Dry_Total_g""","""train/ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,"""Dry_Total_g""",48.2735
"""ID1011485656__GDM_g""","""train/ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,"""GDM_g""",16.275
…,…,…,…,…,…,…,…,…
"""ID983582017__Dry_Clover_g""","""train/ID983582017.jpg""","""2015/9/1""","""WA""","""Ryegrass""",0.64,9.0,"""Dry_Clover_g""",0.0
"""ID983582017__Dry_Dead_g""","""train/ID983582017.jpg""","""2015/9/1""","""WA""","""Ryegrass""",0.64,9.0,"""Dry_Dead_g""",0.0
"""ID983582017__Dry_Green_g""","""train/ID983582017.jpg""","""2015/9/1""","""WA""","""Ryegrass""",0.64,9.0,"""Dry_Green_g""",40.94


In [2]:
train_cleaned = train.select("image_path", "target_name", "target").pivot(index="image_path", on="target_name", values="target")
test_df = test.with_columns(pl.lit(0).alias("target"))
test_cleaned = test_df.select("image_path", "target_name", "target").pivot(index="image_path", on="target_name", values="target")
train_cleaned.head(10)

image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
str,f64,f64,f64,f64,f64
"""train/ID1011485656.jpg""",0.0,31.9984,16.2751,48.2735,16.275
"""train/ID1012260530.jpg""",0.0,0.0,7.6,7.6,7.6
"""train/ID1025234388.jpg""",6.05,0.0,0.0,6.05,6.05
"""train/ID1028611175.jpg""",0.0,30.9703,24.2376,55.2079,24.2376
"""train/ID1035947949.jpg""",0.4343,23.2239,10.5261,34.1844,10.9605
"""train/ID1036339023.jpg""",23.0755,2.6135,32.191,57.88,55.2665
"""train/ID1049634115.jpg""",1.5083,3.0167,13.575,18.1,15.0833
"""train/ID1051144034.jpg""",55.32,0.0,0.0,55.32,55.32
"""train/ID1052620238.jpg""",0.0,11.2291,20.1707,31.3998,20.1707


In [3]:
test_cleaned

image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
str,i32,i32,i32,i32,i32
"""test/ID1001187975.jpg""",0,0,0,0,0


In [4]:
import cv2
import albumentations as A
from tqdm.auto import tqdm

augmented_images_dir = AUGUMENTED_DATA_DIR / "train"
augmented_images_dir.mkdir(exist_ok=True)

# Define transformations
no_transform = A.Compose([])
h_flip = A.Compose([A.HorizontalFlip(p=1.0)])
v_flip = A.Compose([A.VerticalFlip(p=1.0)])
hv_flip = A.Compose([A.HorizontalFlip(p=1.0), A.VerticalFlip(p=1.0)])

transforms = [
    (no_transform, ""),
    (h_flip, "_hflip"),
    (v_flip, "_vflip"),
    (hv_flip, "_hvflip")
]

# Create list to store augmented data
augmented_data = []

# Process each image in train_cleaned
for row in tqdm(train_cleaned.iter_rows(named=True), total=len(train_cleaned)):
    img_path = RAW_DATA_DIR / row['image_path']
    image = cv2.imread(str(img_path))
    
    if image is None:
        continue
    
    # Get the base filename without extension
    base_name = row['image_path'].replace('train/', '').replace('.jpg', '')
    
    for transform, suffix in transforms:
        # Apply transformation
        augmented = transform(image=image)['image']
        
        # Save augmented image
        aug_filename = f"{base_name}{suffix}.jpg"
        aug_path = augmented_images_dir / aug_filename
        cv2.imwrite(str(aug_path), augmented)
        
        if transform == no_transform:
            continue
        
        # Create new row with augmented image path
        new_row = {
            'image_path': f"train/{aug_filename}",
            'Dry_Clover_g': row['Dry_Clover_g'],
            'Dry_Dead_g': row['Dry_Dead_g'],
            'Dry_Green_g': row['Dry_Green_g'],
            'Dry_Total_g': row['Dry_Total_g'],
            'GDM_g': row['GDM_g']
        }
        augmented_data.append(new_row)

# Create new dataframe with augmented data
train_augmented = pl.DataFrame(augmented_data)

# Combine original and augmented data
train_cleaned_expanded = pl.concat([train_cleaned, train_augmented])

print(f"Original dataset size: {len(train_cleaned)}")
print(f"Expanded dataset size: {len(train_cleaned_expanded)}")

  0%|          | 0/357 [00:00<?, ?it/s]

Original dataset size: 357
Expanded dataset size: 1428


In [5]:
train_cleaned_expanded.write_csv(AUGUMENTED_DATA_DIR / "train.csv")
test_cleaned.write_csv(AUGUMENTED_DATA_DIR / "test.csv")

In [6]:
train_cleaned_expanded.sort("image_path")

image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
str,f64,f64,f64,f64,f64
"""train/ID1011485656.jpg""",0.0,31.9984,16.2751,48.2735,16.275
"""train/ID1011485656_hflip.jpg""",0.0,31.9984,16.2751,48.2735,16.275
"""train/ID1011485656_hvflip.jpg""",0.0,31.9984,16.2751,48.2735,16.275
"""train/ID1011485656_vflip.jpg""",0.0,31.9984,16.2751,48.2735,16.275
"""train/ID1012260530.jpg""",0.0,0.0,7.6,7.6,7.6
…,…,…,…,…,…
"""train/ID980878870_vflip.jpg""",32.3575,0.0,2.0325,34.39,34.39
"""train/ID983582017.jpg""",0.0,0.0,40.94,40.94,40.94
"""train/ID983582017_hflip.jpg""",0.0,0.0,40.94,40.94,40.94
